<a href="https://colab.research.google.com/github/Goutham-kc/AIML/blob/main/Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
FILE_NAME = "student_performance.csv"

try:
    df = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    print(f"\nERROR: '{FILE_NAME}' was not found.")
    print("Make sure the CSV file is in the same folder as this program.")
    exit()

print("\nDataset loaded successfully!")
print("Number of records:", len(df))
print("Columns:", df.columns.tolist())

# Features and target
features = [
    "study_hours",
    "attendance",
    "sleep_hours",
    "internet_usage",
    "previous_score",
    "exam_score"
]

target = "placement_status"

# Check columns
missing_columns = [
    column for column in features + [target]
    if column not in df.columns
]

if missing_columns:
    print("\nERROR: Missing columns:")
    for column in missing_columns:
        print("-", column)
    exit()

# Prepare data
X = df[features].copy()
y = df[target].copy()

# Convert features to numbers
for column in features:
    X[column] = pd.to_numeric(X[column], errors="coerce")

# Clean target
y = y.astype(str).str.strip().str.lower()

y = y.map({
    "placed": 1,
    "not placed": 0
})

# Remove invalid target rows
valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

# Fill missing values
X = X.fillna(X.median())

# Check classes
if y.nunique() < 2:
    print("\nERROR: Dataset must contain both 'Placed' and 'Not Placed'.")
    exit()

print("\nClass distribution:")
print(y.value_counts().rename({
    0: "Not Placed",
    1: "Placed"
}))

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train model
model.fit(X_train, y_train)

print("\nModel training completed!")

# Evaluate model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\nMODEL EVALUATION")
print("-" * 40)
print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Placed", "Placed"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Input validation function
def get_number(prompt, minimum, maximum):
    while True:
        value = input(prompt).strip()

        if value.lower() == "q":
            return None

        try:
            value = float(value)

            if minimum <= value <= maximum:
                return value

            print(f"Please enter a value between {minimum} and {maximum}.")

        except ValueError:
            print("Invalid input! Please enter a number.")

# Real-time prediction
print("\n" + "=" * 40)
print("STUDENT PLACEMENT PREDICTOR")
print("=" * 40)
print("Enter 'q' at any time to exit.\n")

while True:
    study_hours = get_number(
        "Study Hours per day (0-24): ",
        0,
        24
    )

    if study_hours is None:
        break

    attendance = get_number(
        "Attendance Percentage (0-100): ",
        0,
        100
    )

    if attendance is None:
        break

    sleep_hours = get_number(
        "Sleep Hours per day (0-24): ",
        0,
        24
    )

    if sleep_hours is None:
        break

    internet_usage = get_number(
        "Internet Usage Hours per day (0-24): ",
        0,
        24
    )

    if internet_usage is None:
        break

    previous_score = get_number(
        "Previous Score (0-100): ",
        0,
        100
    )

    if previous_score is None:
        break

    exam_score = get_number(
        "Exam Score (0-100): ",
        0,
        100
    )

    if exam_score is None:
        break

    student = pd.DataFrame([[
        study_hours,
        attendance,
        sleep_hours,
        internet_usage,
        previous_score,
        exam_score
    ]], columns=features)

    prediction = model.predict(student)[0]
    probabilities = model.predict_proba(student)[0]

    not_placed_probability = probabilities[0]
    placement_probability = probabilities[1]

    print("\n" + "=" * 40)
    print("PREDICTION RESULT")
    print("=" * 40)

    if prediction == 1:
        print("Prediction: PLACED")
    else:
        print("Prediction: NOT PLACED")

    print(f"Placement Probability: {placement_probability:.2%}")
    print(f"Not Placed Probability: {not_placed_probability:.2%}")

    print("\nStudent Information")
    print("-" * 40)
    print(f"Study Hours: {study_hours}")
    print(f"Attendance: {attendance}%")
    print(f"Sleep Hours: {sleep_hours}")
    print(f"Internet Usage: {internet_usage} hours")
    print(f"Previous Score: {previous_score}")
    print(f"Exam Score: {exam_score}")
    print("=" * 40 + "\n")

print("\nProgram terminated.")



Dataset loaded successfully!
Number of records: 10000
Columns: ['study_hours', 'attendance', 'sleep_hours', 'internet_usage', 'assignments_completed', 'previous_score', 'exam_score', 'placement_status']

Class distribution:
placement_status
Placed        8356
Not Placed    1644
Name: count, dtype: int64

Model training completed!

MODEL EVALUATION
----------------------------------------
Accuracy: 99.80%

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.99      1.00      0.99       329
      Placed       1.00      1.00      1.00      1671

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Confusion Matrix:
[[ 328    1]
 [   3 1668]]

STUDENT PLACEMENT PREDICTOR
Enter 'q' at any time to exit.

Study Hours per day (0-24): 4
Attendance Percentage (0-100): 75
Sleep Hours per day (0-24): 8
Internet Usage Hours per day (0-24): 6
P

KeyboardInterrupt: Interrupted by user

In [ ]:
import subprocess
import time
from google.colab.output import eval_js

# Stop any existing Streamlit process
!pkill -f "streamlit run" || true

# Start Streamlit with Colab-compatible settings
subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableXsrfProtection=false",
        "--server.enableCORS=false"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Get the Colab proxy URL
url = eval_js("google.colab.kernel.proxyPort(8501)")

print("Open your Streamlit app:")
print(url)

^C
Open your Streamlit app:
https://8501-m-s-kkb-ase1a2-hgzofq75zpud-a.asia-east1-2.prod.colab.dev
